In [9]:
# Libraries
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Layer, LeakyReLU, ReLU, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.initializers import HeNormal, GlorotNormal
from tensorflow.keras.callbacks import LearningRateScheduler
from tensorflow.keras.regularizers import l1_l2
import numpy as np

from keras.regularizers import l1, l2
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [4]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import pandas as pd
import json
# Import config variable
with open("../../config.json") as f:
    cfg = json.load(f)
    
target = cfg["rc"]        
file_name = cfg["rc_file_name"]   
random_state = cfg["split"]["random_state"]  
# Load the dataset
data = pd.read_csv(f"../../data/{file_name}.csv")
# data = pd.read_csv(f"../../data/dataset_dropna.csv")

In [5]:
X = data.drop(columns=[target])
y = data[target]

# X = data.drop(columns=['Rc (MPa)'])
# y = data['Rc (MPa)']

X_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Tirage de 4 seeds aléatoires différentes
master_rng = np.random.default_rng(seed=99)
seeds = master_rng.choice(10000, size=4, replace=False).tolist()
print("Seeds sélectionnées :", seeds)

# Split fixe (random_state=42) utilisé par la grille de recherche ci-dessous
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),      columns=X_test.columns)

Seeds sélectionnées : [5650, 7575, 5059, 9578]


In [ ]:
from tensorflow.keras.initializers import HeNormal, GlorotNormal

k_folds_train = 5
kf = KFold(n_splits=k_folds_train, shuffle=True, random_state=42)

initGlorotNormal = tf.keras.initializers.GlorotNormal()
l1_strength = 0.001
l2_strength = 0.01

early_stopping = EarlyStopping(patience=20, restore_best_weights=True)

mse_scores_train = []
mse_scores_val = []


r_squareds_train_all = []

r_squareds_val_all = []

models = []
historys = []

r_batch = []
batch_sizes = [4, 7, 10, 13, 16]
# batch = 14
# lr = 0.001
learning_rates = [0.0003, 0.0005, 0.0008, 0.001, 0.002]

r_squareds_train_lr = []
r_squareds_val_lr = []
mse_scores_train_lr = []
mse_scores_val_lr = []

for lr in learning_rates:
    
    for batch in batch_sizes :
        r_squareds_train_batch = []
        r_squareds_val_batch = []
        mse_scores_train_batch = []
        mse_scores_val_batch = []

        ann_rf = Sequential([
        Dense(32, activation= 'relu', kernel_initializer=initGlorotNormal),
        Dense(16, activation= 'relu', kernel_initializer=initGlorotNormal),
        Dropout(0.2),
        Dense(1, activation='linear', kernel_regularizer= l2(l2_strength))
        ])

        optimizer = tf.keras.optimizers.Adam(learning_rate= lr, decay = 0.01) # , decay = 0.01

        ann_rf.compile(optimizer=optimizer, 
                loss= 'mse', 
                metrics=['mae'])

        for train_index, val_index in kf.split(X_train_scaled):
            X_train_fold, X_val_fold = X_train_scaled.iloc[train_index], X_train_scaled.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            # lr_scheduler_callback = LearningRateScheduler(lr_scheduler)

            history_train = ann_rf.fit(X_train_fold, y_train_fold, epochs=100, batch_size=batch, verbose=0)
            historys.append(history_train)

            y_pred_train = ann_rf.predict(X_train_fold)
            r_squared_train = r2_score(y_train_fold, y_pred_train)
            r_squareds_train_batch.append(r_squared_train)
            mse_train = mean_squared_error(y_train_fold, y_pred_train)
            mse_scores_train_batch.append(mse_train)

            y_pred_val = ann_rf.predict(X_val_fold)
            r_squared_val = r2_score(y_val_fold, y_pred_val)
            r_squareds_val_batch.append(r_squared_val)
            mse_val = mean_squared_error(y_val_fold, y_pred_val)
            mse_scores_val_batch.append(mse_val)

        r_squareds_train_lr.append(r_squareds_train_batch)
        r_squareds_val_lr.append(r_squareds_val_batch)
        mse_scores_train_lr.append(mse_scores_train_batch)
        mse_scores_val_lr.append(mse_scores_val_batch)
        models.append(ann_rf)
        print('Model indice :', len(models)-1)

        print('With Learning rate =', lr,"With batch =", batch, "R²_train =", np.mean(r_squareds_train_batch))
        print('With Learning rate =', lr,"With batch =", batch, "mse_train =", np.mean(mse_scores_train_batch))
        print('')
        print('With Learning rate =', lr,"With batch =", batch, "R²_val =", np.mean(r_squareds_val_batch))
        print('With Learning rate =', lr,"With batch =", batch, "mse_val =", np.mean(mse_scores_val_batch))
        print('')
        y_pred_test= ann_rf.predict(X_test_scaled)
        # y_pred_test = y_scaler.inverse_transform(y_pred_test_scaled)
        r2_rc = r2_score(y_test, y_pred_test)

        print('With Learning rate =', lr, 'With batch =', batch, "R-squared on Test Set:", r2_rc)
            #MSE
        mae_rc = mean_absolute_error(y_test, y_pred_test)

        mse_test = mean_squared_error(y_test, y_pred_test)

        print('With Learning rate =', lr,'With batch =', batch, 'Mean Absolute Error test:', mae_rc)
        print('With Learning rate =', lr,'With batch =', batch, 'Mean Squared Error test:', mse_test)
        print('With Learning rate =', lr,'With batch =', batch, 'Root Mean Squared Error test:', np.sqrt(mse_test))
        print('')
        print('With Learning rate =', lr,'With batch_size = ', batch, 'Act vs Pred :')

C:\Users\matte\AppData\Roaming\Python\Python313\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
Model indice : 0
With Learning rate = 0.0003 With batch = 4 R²_train = 0.9781236052513123
With Learning rate = 0.0003 With batch = 4 mse_train = 1.423589277267456

With Learning rate = 0.0003 With batch = 4 R²_val = 0.9619146585464478
With Learning rate = 0.0003 With batch = 4 mse_val = 2.3215787410736084

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
With Learning rate = 0.0003 With batch = 4 R-squared on Test Set: 0.9454968571662903
With Learning rate = 0.0003 With batch = 4 Mean Absolute Error test: 1.3594374656677246
With Learning rate = 0.0003 With batch = 4 Mean Squared Error test: 3.8684263229370117
With Learning 

C:\Users\matte\AppData\Roaming\Python\Python313\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
Model indice : 1
With Learning rate = 0.0003 With batch = 7 R²_train = 0.9672897577285766
With Learning rate = 0.0003 With batch = 7 mse_train = 2.127824091911316

With Learning rate = 0.0003 With batch = 7 R²_val = 0.9421522498130799
With Learning rate = 0.0003 With batch = 7 mse_val = 3.514656591415405

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
With Learning rate = 0.0003 With batch = 7 R-squared on Test Set: 0.9255673885345459
With Learning rate = 0.0003 With batch = 7 Mean Absolute Error test: 1.6540149450302124
With Learning rate = 0.0003 With batch = 7 Mean Squared Error test: 5.282945156097412
With Learning ra

C:\Users\matte\AppData\Roaming\Python\Python313\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
Model indice : 2
With Learning rate = 0.0003 With batch = 10 R²_train = 0.9635967373847961
With Learning rate = 0.0003 With batch = 10 mse_train = 2.3704050302505495

With Learning rate = 0.0003 With batch = 10 R²_val = 0.9485058903694152
With Learning rate = 0.0003 With batch = 10 mse_val = 3.143825078010559

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
With Learning rate = 0.0003 With batch = 10 R-squared on Test Set: 0.9573977589607239
With Learning rate = 0.0003 With batch = 10 Mean Absolute Error test: 1.4056330919265747
With Learning rate = 0.0003 With batch = 10 Mean Squared Error test: 3.0237462520599365
With Le

C:\Users\matte\AppData\Roaming\Python\Python313\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
import joblib
import os

os.makedirs('saved_models', exist_ok=True)

# Sauvegarde du meilleur ANN (le dernier entraîné ici, adapte si tu veux le meilleur)
ann_rf.save(f'saved_models/ann_lr{lr}_batch{batch}.keras')

# Sauvegarde du scaler (indispensable pour réutiliser le modèle)
joblib.dump(scaler, 'saved_models/scaler_rc.pkl')

NameError: name 'ann_rf' is not defined

In [ ]:
import joblib
import os

os.makedirs('saved_models', exist_ok=True)

lr            = 0.002   # meilleur lr trouvé dans la grille
batch         = 16      # meilleur batch trouvé dans la grille
k_folds_train = 5
l2_strength   = 0.01

results_per_seed = {}

for seed in seeds:
    print(f"\n{'='*50}")
    print(f"Entraînement avec seed = {seed}")
    print(f"{'='*50}")

    # --- Split & scaling propre pour cette seed ---
    X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )
    scaler_s = StandardScaler()
    X_train_scaled_s = pd.DataFrame(scaler_s.fit_transform(X_train_s), columns=X_train_s.columns)
    X_test_scaled_s  = pd.DataFrame(scaler_s.transform(X_test_s),      columns=X_test_s.columns)

    # --- KFold ---
    kf_s = KFold(n_splits=k_folds_train, shuffle=True, random_state=seed)

    # --- Modèle ---
    tf.random.set_seed(seed)
    np.random.seed(seed)

    model_s = Sequential([
        Dense(32, activation='relu', kernel_initializer=GlorotNormal(seed=seed)),
        Dense(16, activation='relu', kernel_initializer=GlorotNormal(seed=seed)),
        Dropout(0.2),
        Dense(1,  activation='linear', kernel_regularizer=l2(l2_strength))
    ])
    model_s.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr, decay=0.01),
        loss='mse', metrics=['mae']
    )

    r2_train_folds, r2_val_folds = [], []

    for train_idx, val_idx in kf_s.split(X_train_scaled_s):
        X_tr, X_vl = X_train_scaled_s.iloc[train_idx], X_train_scaled_s.iloc[val_idx]
        y_tr, y_vl = y_train_s.iloc[train_idx],        y_train_s.iloc[val_idx]

        model_s.fit(X_tr, y_tr, epochs=100, batch_size=batch, verbose=0,
                    callbacks=[EarlyStopping(patience=20, restore_best_weights=True)])

        r2_train_folds.append(r2_score(y_tr,  model_s.predict(X_tr)))
        r2_val_folds.append(  r2_score(y_vl,  model_s.predict(X_vl)))

    y_pred_test_s = model_s.predict(X_test_scaled_s)
    r2_test  = r2_score(y_test_s, y_pred_test_s)
    mae_test = mean_absolute_error(y_test_s, y_pred_test_s)
    mse_test = mean_squared_error(y_test_s, y_pred_test_s)

    print(f"R² train (moy folds) : {np.mean(r2_train_folds):.4f}")
    print(f"R² val   (moy folds) : {np.mean(r2_val_folds):.4f}")
    print(f"R² test              : {r2_test:.4f}")
    print(f"MAE test             : {mae_test:.4f}")
    print(f"RMSE test            : {np.sqrt(mse_test):.4f}")

    results_per_seed[seed] = {
        "r2_train": np.mean(r2_train_folds),
        "r2_val":   np.mean(r2_val_folds),
        "r2_test":  r2_test,
        "mae_test": mae_test,
        "rmse_test": np.sqrt(mse_test),
    }

    # --- Export modèle + scaler ---
    model_s.save(f'saved_models/ann_seed_{seed}.keras')
    joblib.dump(scaler_s, f'saved_models/scaler_seed_{seed}.pkl')
    print(f"  -> saved_models/ann_seed_{seed}.keras  +  scaler_seed_{seed}.pkl")

print("\n\n=== Récapitulatif ===")
for s, m in results_per_seed.items():
    print(f"seed={s:5d}  R²_test={m['r2_test']:.4f}  MAE={m['mae_test']:.4f}  RMSE={m['rmse_test']:.4f}")


Entraînement avec seed = 5650


C:\Users\matte\AppData\Roaming\Python\Python313\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(
C:\Users\matte\AppData\Roaming\Python\Python313\site-packages\keras\src\callbacks\early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,mae
  current = self.get_monitor_value(logs)


8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step


C:\Users\matte\AppData\Roaming\Python\Python313\site-packages\keras\src\callbacks\early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,mae
  current = self.get_monitor_value(logs)


8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step


C:\Users\matte\AppData\Roaming\Python\Python313\site-packages\keras\src\callbacks\early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,mae
  current = self.get_monitor_value(logs)


... (31 outputs left out)

C:\Users\matte\AppData\Roaming\Python\Python313\site-packages\keras\src\callbacks\early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,mae
  current = self.get_monitor_value(logs)


8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step


C:\Users\matte\AppData\Roaming\Python\Python313\site-packages\keras\src\callbacks\early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,mae
  current = self.get_monitor_value(logs)


8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
R² train (moy folds) : 0.7722
R² val   (moy folds) : 0.5681
R² test              : 0.7916
MAE test             : 2.9422
RMSE test            : 3.9324
  -> saved_models/ann_seed_9578.keras  +  scaler_seed_9578.pkl


=== Récapitulatif ===
seed= 5650  R²_test=0.8180  MAE=2.4799  RMSE=3.4618
seed= 7575  R²_test=0.6804  MAE=3.1345  RMSE=4.3855
seed= 5059  R²_test=0.8614  MAE=2.3485  RMSE=3.0955
seed= 9578  R²_test=0.7916  MAE=2.9422  RMSE=3.9324
